# Hybrid RAG for Personalized Skincare Recommendations

This notebook builds and evaluates a **Hybrid RAG** recommender that combines:
- **Vector search** over review semantics (baseline)
- **Graph signals** from a Neo4j Knowledge Graph (brand loyalty re-ranking)

Outputs:
- Side-by-side **Vector-only vs Hybrid** Top-K recommendations
- A short **rationale** explaining why the Hybrid ranking changed
- Quantitative (Hit Rate@10, NDCG@10)  Qualitative (Ragas) evaluation.

## Quickstart

1. Create a `.env` file from `.env.example` (recommended).  
   If not set, the notebook will prompt via `getpass()` at runtime.
2. Ensure `./data` contains:
   - `amazon_face_core_meta.csv`
   - `amazon_face_core_reviews.csv`
3. Run cells top-to-bottom.

## Method

### Data
Subset of **Amazon Reviews 2023** (Facial Skincare), filtered to "core users" with sufficient purchases.

### Algorithm
1. **Candidate Generation:** Vector search retrieves top-N review-based candidates.
2. **Dynamic Re-ranking:** Apply a **log-weighted boost** based on user-brand history in the KG.

**Scoring:**
\[
FinalScore = VectorScore \times (1  \log(1  HistoryCount) \times w)
\]


## Problem & Approach

**Problem:** Vector search retrieves semantically similar products, but does not reflect user-specific preferences (e.g., brand loyalty).  
**Solution:** Re-rank vector candidates by injecting a **log-weighted brand loyalty signal** from a Neo4j Knowledge Graph.

**Core idea:** Use vector search for candidate generation, then apply a graph-based boost as a tie-breaker for ambiguous queries.

---
*See below for detailed implementation and code execution.*

## Technical Implementation
### Step 1. Environment & Credentials
- Load environment variables (`.env`) or prompt via `getpass()`

In [ ]:
# =====================================================
# Standard library
# =====================================================

import os
import time
import random

# Third-party
import numpy as np
import pandas as pd
from tqdm import tqdm

from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_similarity,
)
from datasets import Dataset

# Make src importable (notebook-friendly)
import sys
from pathlib import Path

repo_root = Path.cwd()
# If notebook is executed from notebooks/, go one level up
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
# =====================================================
# Global configuration
# =====================================================

load_dotenv()  # Loads .env from the current working directory (if present)

# Secrets: prefer env vars; fallback to interactive prompt
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set. Add it to .env or export it in your shell.")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Models: env override supported
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "gpt-4.1-mini")

# Clients (kept consistent with your current notebook usage)
client = OpenAI(api_key=OPENAI_API_KEY)
llm_service = ChatOpenAI(model=CHAT_MODEL)
llm_judge = ChatOpenAI(model=JUDGE_MODEL)
emb = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print(f"Models configured: service={CHAT_MODEL}, judge={JUDGE_MODEL}, embed={EMBEDDING_MODEL}")

In [ ]:
# ============================================================
# Neo4j connection (Global Driver) - Load from .env or prompt
# ============================================================

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://172.31.240.1:7691")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD is not set. Add it to .env or export it in your shell.")
    
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Neo4j connected: {NEO4J_URI}")

### Step 2. Load Data
- Read product metadata and review corpus from `./data`

In [ ]:
# =======================
# Data Loading Paths
# =======================
BASE_PATH = "./data" 
META_FILE = os.path.join(BASE_PATH, "amazon_face_core_meta.csv")
REVIEW_FILE = os.path.join(BASE_PATH, "amazon_face_core_reviews.csv")

### Step 3. Build Neo4j Graph
- Create nodes/relationships needed for:
  - Users
  - Products (with brand)
  - Purchase / interaction history

In [ ]:
# ==============================================
# Helper Functions: import products and reviews
# ==============================================

def init_neo4j_schema(driver):
    """
    Initialize database constraints to ensure data integrity and performance.
    Creates unique constraints on User ID, Product ID, Brand Name, etc.
    """
    print("Initializing Database Schema & Constraints...")
    queries = [
        "CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE",
        "CREATE CONSTRAINT product_id IF NOT EXISTS FOR (p:Product) REQUIRE p.asin IS UNIQUE",
        "CREATE CONSTRAINT review_id IF NOT EXISTS FOR (r:Review) REQUIRE r.id IS UNIQUE",
        "CREATE CONSTRAINT brand_name IF NOT EXISTS FOR (b:Brand) REQUIRE b.name IS UNIQUE",
        "CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE"
    ]
    
    try:
        with driver.session() as session:
            for q in queries:
                session.run(q)
        print("Schema Constraints Created Successfully.")
    except Exception as e:
        print(f"Schema Initialization Warning: {e}")

def import_products(driver):
    """
    Import Product, Brand, and Category nodes from metadata CSV.
    """
    if not os.path.exists(META_FILE):
        print(f"Error: Meta file not found: {META_FILE}")
        return

    print(f"\nImporting Products from: {META_FILE}")
    
    df_meta = pd.read_csv(META_FILE)
    df_meta.fillna({'brand': 'Unknown', 'title': 'No Title', 'price': 0}, inplace=True)
    
    print(f"   - Total Products to Import: {len(df_meta):,}")
    
    # Cypher Query: Create Product, Brand, Category and their relationships
    query = """
    UNWIND $rows AS row
    MERGE (p:Product {asin: row.parent_asin})
    SET p.title = row.title, 
        p.price = toFloat(row.price), 
        p.features = row.features
        
    MERGE (b:Brand {name: row.brand})
    MERGE (p)-[:MADE_BY]->(b)
    
    MERGE (c:Category {name: 'Face'})
    MERGE (p)-[:BELONGS_TO]->(c)
    """
    
    # Batch Processing
    batch_size = 1000
    with driver.session() as session:
        for i in tqdm(range(0, len(df_meta), batch_size), desc="Processing Batches"):
            batch = df_meta.iloc[i:ibatch_size].to_dict('records')
            session.run(query, rows=batch)
            
    print("Product Import Completed.")

def import_reviews(driver):
    """
    Import User and Review nodes, creating relationships between Users, Reviews, and Products.
    """
    if not os.path.exists(REVIEW_FILE):
        print(f"Error: Review file not found: {REVIEW_FILE}")
        return

    print(f"\nImporting Reviews from: {REVIEW_FILE}")
    df_review = pd.read_csv(REVIEW_FILE)
    
    df_review["review_id"] = df_review["user_id"].astype(str)  "_"  df_review["parent_asin"].astype(str)
    
    print(f"   - Total Reviews to Import: {len(df_review):,}")
    
    # Cypher Query: Create User, Review and link them
    query = """
    UNWIND $rows AS row
    
    MERGE (u:User {user_id: row.user_id})
    
    MERGE (r:Review {id: row.review_id})
    SET r.rating = toInteger(row.rating), 
        r.text = row.text, 
        r.timestamp = row.timestamp, 
        r.helpful_vote = toInteger(row.helpful_vote)
        
    MERGE (u)-[:WROTE]->(r)
    
    WITH r, row
    MATCH (p:Product {asin: row.parent_asin})
    MERGE (r)-[:EVALUATED]->(p)
    """
    
    batch_size = 1000
    with driver.session() as session:
        for i in tqdm(range(0, len(df_review), batch_size), desc="Processing Batches"):
            batch = df_review.iloc[i:ibatch_size].to_dict('records')
            session.run(query, rows=batch)
            
    print("Review Import Completed.")

In [ ]:
# ==========================================
# Execution
# ==========================================
init_neo4j_schema(driver)
import_products(driver)
import_reviews(driver)

print("✔ All Neo4j imports completed successfully.")

### Step 4. Vector Indexing & Embeddings
- Generate embeddings for reviews
- Build Neo4j vector index (`review_embedding_index`)

In [ ]:
# ==========================================
# Vector_indexing: Configuration
# ==========================================

EMBEDDING_DIMENSION = 1536
MAX_EMBEDDING_CHARS = 2000
BATCH_SIZE = 1000  # Adjust based on API rate limits

In [ ]:
# ==========================================
# Helper Functions
# ==========================================

def get_reviews_without_embeddings(driver, limit=1000):
    """
    Fetch reviews that do not have an embedding property yet.
    """
    query = """
    MATCH (r:Review)
    WHERE r.embedding IS NULL AND r.text IS NOT NULL
    RETURN r.id AS id, r.text AS text
    LIMIT $limit
    """
    with driver.session() as session:
        result = session.run(query, limit=limit)
        return [{"id": record["id"], "text": record["text"]} for record in result]

def update_review_embeddings(driver, updates):
    """
    Update Review nodes using standard SET query.
    """
    query = """
    UNWIND $updates AS row
    MATCH (r:Review {id: row.id})
    SET r.embedding = row.embedding
    """
    try:
        with driver.session() as session:
            session.run(query, updates=updates)
    except Exception as e:
        print(f"Critical Error updating embeddings: {e}")
        raise e

def create_vector_index(driver):
    """
    Create Vector Index if it doesn't exist.
    """
    index_name = "review_embedding_index"
    print(f"Creating/Verifying Vector Index: {index_name}...")
    
    check_query = "SHOW INDEXES WHERE name = $name"
    with driver.session() as session:
        result = session.run(check_query, name=index_name)
        if result.peek():
            print(f"✔ Index '{index_name}' already exists. Skipping creation.")
            return

    create_query = f"""
    CREATE VECTOR INDEX {index_name} IF NOT EXISTS
    FOR (r:Review)
    ON (r.embedding)
    OPTIONS {{indexConfig: {{
      `vector.dimensions`: 1536,
      `vector.similarity_function`: 'cosine'
    }}}}
    """
    try:
        with driver.session() as session:
            session.run(create_query)
        print(f"✔ Successfully created vector index: {index_name}")
        print("Waiting for index to be online...")
        time.sleep(5)
    except Exception as e:
        print(f"⚠ Failed to create index: {e}")

def generate_embeddings(text_list):
    """
    Generate embeddings using the global OpenAI client.
    """
    try:
        clean_texts = []
        for text in text_list:
            if not isinstance(text, str):
                text = str(text)
            text = text.replace("\n", " ")
            text = text[:MAX_EMBEDDING_CHARS] 
            clean_texts.append(text)

        # Use global 'client' and 'EMBEDDING_MODEL'
        response = client.embeddings.create(
            input=clean_texts,
            model=EMBEDDING_MODEL
        )
        return [data.embedding for data in response.data]

    except Exception as e:
        print(f"OpenAI API Error: {e}")
        return []

In [ ]:
# ==========================================
# Execution: Embedding & Indexing
# ==========================================
try:
    print("\nStarting Full Vector Embedding Generation...")
    total_processed = 0
    
    # Use global 'driver' directly
    while True:
        # Fetch batch
        batch = get_reviews_without_embeddings(driver, limit=BATCH_SIZE)
        
        if not batch:
            print("✔ No more reviews to process. All embeddings generated.")
            break
        
        texts = [item["text"] for item in batch]
        ids = [item["id"] for item in batch]
        
        print(f"Generating embeddings for batch of {len(texts)} reviews...")
        start_time = time.time()
        
        # API Call (using global client inside function)
        embeddings = generate_embeddings(texts)
        
        if not embeddings:
            print("Failed to generate embeddings. Stopping process.")
            break
            
        # Update DB
        update_data = [{"id": uid, "embedding": emb} for uid, emb in zip(ids, embeddings)]
        update_review_embeddings(driver, update_data)
        
        total_processed = len(batch)
        elapsed = time.time() - start_time
        print(f"   - Processed {total_processed} reviews successfully. (Batch time: {elapsed:.2f}s)")

    # Create Index after processing
    print("\nBuilding Vector Index...")
    create_vector_index(driver)
    
    print("\n✔ All Embedding Steps Completed Successfully!")

except Exception as e:
    print(f"\nError: {e}")

### Step 5. Hybrid Retrieval
- Core algorithm implemented to fuse semantic similarity with domain-specific graph signals.
- run_hybrid_query defined to combine vector scores with log-weighted brand loyalty scores.
- generate_answer implemented to synthesize personalized recommendations using the LLM.

In [ ]:
# ====================================================================
# Helper Functions: Dynamic Brand Loyalty Weighting & Hybrid Retrieval
# ====================================================================

def get_embedding(text):
    """
    Generate embedding vector using Global Client.
    """
    clean = str(text).replace("\n", " ")
    # Use global 'client' and 'EMBEDDING_MODEL'
    return client.embeddings.create(
        input=[clean], 
        model=EMBEDDING_MODEL
    ).data[0].embedding
    
def analyze_user_loyalty(driver, user_id):
    """
    Returns:
        loyalty_score (float): brand_count / total_count (fallback 0.0)
        top_brand (str|None)
        purchased_asins (list[str])
    """
    query = """
    CALL {
      MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(:Product)-[:MADE_BY]->(:Brand)
      RETURN count(*) AS total_count
    }
    MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(p:Product)-[:MADE_BY]->(b:Brand)
    WITH total_count, b, count(*) AS brand_count, collect(DISTINCT p.asin) AS products
    ORDER BY brand_count DESC
    RETURN total_count, b.name AS brand_name, brand_count, products
    LIMIT 1
    """
    with driver.session() as session:
        row = session.run(query, user_id=user_id).single()
        if not row:
            return 0.0, None, []

        total = int(row["total_count"] or 0)
        brand_count = int(row["brand_count"] or 0)
        products = row["products"] or []

        if total <= 0:
            return 0.0, row["brand_name"], products

        loyalty_score = brand_count / total
        return float(loyalty_score), row["brand_name"], products

In [ ]:
# Dynamic Hybrid Retrieval

def run_hybrid_query(
    driver,
    query_embedding,
    user_id=None,
    loyalty_score=0.0,
    k=5,
    exclude_purchased=False,
    return_context_string=False,
):
    """
    Hybrid retrieval = vector search  brand-history boost (GraphRAG).
    Returns list[dict] by default; if return_context_string=True returns list[str].
    """
    brand_weight = float(loyalty_score) * 0.5

    filter_clause = ""
    if exclude_purchased and user_id:
        filter_clause = """
        WHERE NOT EXISTS {
          MATCH (u:User {user_id: $user_id})-[:WROTE]->(:Review)-[:EVALUATED]->(product)
        }
        """

    cypher_query = f"""
    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)
    YIELD node AS similar_review, score AS vector_score

    MATCH (similar_review)-[:EVALUATED]->(product:Product)-[:MADE_BY]->(brand:Brand)
    {filter_clause}

    OPTIONAL MATCH (u:User {{user_id: $user_id}})-[:WROTE]->(:Review)-[:EVALUATED]->(:Product)-[:MADE_BY]->(brand)
    WITH product, brand, similar_review, vector_score, count(u) AS history_count

    WITH product, brand, vector_score, history_count, similar_review,
         (vector_score * (1  (log(1  history_count) * $brand_weight))) AS final_score

    ORDER BY final_score DESC
    LIMIT $k

    RETURN product.title AS product_name,
           product.asin  AS asin,
           brand.name    AS brand_name,
           product.features AS features,
           similar_review.text AS review_text,
           vector_score,
           history_count,
           final_score
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            embedding=query_embedding,
            user_id=user_id,
            brand_weight=brand_weight,
            k=int(k),
        )
        records = [r.data() for r in result]

    if return_context_string:
        return [f"{x.get('product_name','')} by {x.get('brand_name','')}: {x.get('review_text','')}" for x in records]
    return records

In [ ]:
def generate_answer(client, query, context, user_id=None, top_brand=None):
    """
    Generate answer using LLM based on retrieved context.
    """
    system_prompt = """
    You are an expert Personal Shopper AI.
    Recommend products based on the user's query and the retrieved candidates.
    
    Analysis of User Style:
    - If the user has a 'Favorite Brand', acknowledge it but also introduce high-quality alternatives.
    - Explain clearly why each product fits their specific need (based on features/reviews).
    """
    
    context_text = ""
    for i, item in enumerate(context):
        # Handle case where context might be simple strings (RAGAS) or dicts (Engine)
        if isinstance(item, str):
            context_text = f"\n[Candidate #{i1}] {item}"
        else:
            context_text = f"""
            [Candidate #{i1}]
            - Product: {item.get('product_name')}
            - Brand: {item.get('brand_name')}
            - Relevance Score: {item.get('final_score'):.4f}
            - User's Brand History: {item.get('history_count')} times purchased
            - Key Review Snippet: "{item.get('review_text', '')[:100]}..."
            """
    
    user_prompt = f"""
    User Query: {query}
    User ID: {user_id}
    User's Favorite Brand: {top_brand if top_brand else "None"}
    
    [Market Data / Context]:
    {context_text}
    """

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content


### Step 6. Case Study (Dynamic Personalization)
- Same query, different user IDs → different rankings via graph signal

In [ ]:
# ==========================================
# Case Study: Qualitative Evaluation
# ==========================================

# Target User ID provided by you
TARGET_USER_ID = "AEXCLMGS3Y7SRW5CMLJBNYI2HBZQ"

def get_user_last_review_text(tx, user_id):
    """
    Fetch the last review written by the user to use as a query.
    """
    query = """
    MATCH (u:User {user_id: $user_id})-[:WROTE]->(r:Review)-[:EVALUATED]->(p:Product)
    RETURN r.text AS text, p.title AS product
    ORDER BY r.timestamp DESC
    LIMIT 1
    """
    result = tx.run(query, user_id=user_id).single()
    return (result['text'], result['product']) if result else (None, None)

In [ ]:
# ==========================================
# Execution: Recommendation Demo (Clean)
# ==========================================

TOP_K = 10
LOYALTY_SCORE = 0.2
EXCLUDE_PURCHASED = False

with driver.session() as session:
    review_text, last_product = session.execute_read(get_user_last_review_text, TARGET_USER_ID)

if not review_text:
    print("User has no reviews found.")
else:
    query_vec = get_embedding(review_text)
    rec_results = run_hybrid_query(
        driver,
        query_embedding=query_vec,
        user_id=TARGET_USER_ID,
        loyalty_score=LOYALTY_SCORE,
        k=TOP_K,
        exclude_purchased=EXCLUDE_PURCHASED,
        return_context_string=False,
    )

    print(f"[Ground Truth] {last_product}")
    print(f"{'Brand':<20} | {'History':<7} | {'VecScore':<9} | {'FinalScore':<10} | Product")
    print("-" * 105)
    for r in rec_results:
        history = int(r.get("history_count", 0) or 0)
        mark = "*" if history > 0 else " "
        brand = str(r.get("brand_name", ""))[:18]
        vec = float(r.get("vector_score", 0.0) or 0.0)
        fin = float(r.get("final_score", 0.0) or 0.0)
        prod = str(r.get("product_name", ""))[:55]
        print(f"{mark} {brand:<18} | {history:<7} | {vec:<9.4f} | {fin:<10.4f} | {prod}...")

### Step 7. Evaluation: Quantitative Performance Metrics
* Hit Rate@10 and NDCG@10 measured on a test set of 50 users:
    * **Hit Rate@10:** Measures whether the target product appears within the top-10 results. (Formula: $1$ if $Target \in Top10$ else $0$)
    * **NDCG@10:** Evaluates ranking quality by prioritizing correct items at higher positions. (Formula: $1 / \log_2(Rank  1)$)

In [ ]:
# ==========================================
# Test Settings
# ==========================================

SAMPLE_SIZE = 50  
TOP_K = 10        # Top-10 recommendations

In [ ]:
# ==========================================
# Helper Functions: Evaluation
# ==========================================

def fetch_test_dataset(driver, sample_size=50):
    """
    Fetch 'Core Users' and their 'Last Review' to create a Ground Truth dataset.
    """
    print(f"Building Test Dataset (Sample: {sample_size})...")
    query = """
    MATCH (u:User)-[:WROTE]->(r:Review)-[:EVALUATED]->(p:Product)
    WITH u, count(r) AS review_count
    WHERE review_count >= 5
    WITH u
    ORDER BY rand()
    LIMIT $limit
    CALL {
      WITH u
      MATCH (u)-[:WROTE]->(r2:Review)-[:EVALUATED]->(p2:Product)
      RETURN r2.text AS raw_review_text, p2.title AS target_product
      ORDER BY coalesce(r2.timestamp, r2.unix_time, r2.time, r2.created_at) DESC
      LIMIT 1
    }
    RETURN u.user_id AS user_id, raw_review_text, target_product
    """
    with driver.session() as session:
        rows = session.run(query, limit=sample_size).data()

    cases = []
    for row in rows:
        uid = row.get("user_id")
        raw = row.get("raw_review_text") or ""
        tgt = row.get("target_product") or ""
        loyalty, _, _ = analyze_user_loyalty(driver, uid) if uid else (0.0, None, [])
        cases.append({
            "user_id": uid,
            "raw_review_text": raw,
            "target_product": tgt,
            "loyalty_score": float(loyalty),
        })
    return cases

def generate_synthetic_query(raw_review_text: str) -> str:
    """
    Convert a specific product review into a generic shopping query (reduce leakage).
    """
    prompt = (
        "Rewrite the following product review into a short, generic shopping query that describes the user's need. "
        "Do NOT mention brand names or product names. Output one sentence.\n\n"
        f"Review:\n{raw_review_text}"
    )
    msg = llm_service.invoke(prompt)
    return (msg.content or "").strip()

def run_search(driver, query_embedding, user_id=None, loyalty_score=0.0, mode="vector", k=10):
    """
    mode:
      - 'vector': vector-only retrieval (product titles)
      - 'hybrid': run_hybrid_query re-ranking (product titles)
    """
    if mode == "hybrid":
        recs = run_hybrid_query(
            driver,
            query_embedding=query_embedding,
            user_id=user_id,
            loyalty_score=loyalty_score,
            k=k,
            exclude_purchased=False,
            return_context_string=False,
        )
        return [r.get("product_name", "") for r in recs]

    cypher = """
    CALL db.index.vector.queryNodes('review_embedding_index', 150, $embedding)
    YIELD node AS similar_review, score AS vector_score
    MATCH (similar_review)-[:EVALUATED]->(p:Product)
    RETURN p.title AS product_name, vector_score
    ORDER BY vector_score DESC
    LIMIT $k
    """
    with driver.session() as session:
        rows = session.run(cypher, embedding=query_embedding, k=k).data()
    return [r["product_name"] for r in rows]

In [ ]:
# ==========================================
# Quantitative Evaluation: Execution Loop
# ==========================================

print(f"Connecting to Neo4j at {NEO4J_URI} (Using Global Driver)...")
driver.verify_connectivity()

# Prepare Test Data
test_data = fetch_test_dataset(driver, sample_size=SAMPLE_SIZE)
print(f"Fetched {len(test_data)} raw test cases.")

# Pre-processing: Generate Synthetic Queries
print("\n Generating Synthetic Queries (Paraphrasing) to reduce data leakage...")
print("   (Converting specific reviews into general user needs...)")

for case in tqdm(test_data, desc="Paraphrasing"):
    # Convert raw review -> Generic Search Query
    case['query_text'] = generate_synthetic_query(case['raw_review_text'])

# Show an example of the transformation
print(f"\n[Example Transformation]")
print(f"Original Review: {test_data[0]['raw_review_text'][:80]}...")
print(f"Generated Query: {test_data[0]['query_text']}")
print("-" * 50)

quant_scores = {
     'vector': {'hit': [], 'ndcg': []},
     'hybrid': {'hit': [], 'ndcg': []}
 }

print("\nStarting Evaluation (Comparing Vector vs. Hybrid)...")

# Evaluation Loop
try:
    for case in tqdm(test_data, desc="Evaluating"):
        user_id = case['user_id']
        target = case['target_product']
        loyalty = case['loyalty_score']
        query_text = case['query_text'] # Uses the NEW synthetic query

        # Use global client inside get_embedding
        query_vec = get_embedding(query_text)

        # Vector Only Search
        recs_vector = run_search(driver, query_vec, user_id, loyalty, mode="vector", k=10)
        h_v, n_v = calculate_metrics(recs_vector, target)
        quant_scores['vector']['hit'].append(h_v)
        quant_scores['vector']['ndcg'].append(n_v)

        # Hybrid Search
        recs_hybrid = run_search(driver, query_vec, user_id, loyalty, mode="hybrid", k=10)
        h_h, n_h = calculate_metrics(recs_hybrid, target)
        quant_scores['hybrid']['hit'].append(h_h)
        quant_scores['hybrid']['ndcg'].append(n_h)
        
    print("\n✔ Evaluation Loop Completed.")

except Exception as e:
    print(f"\nError during evaluation loop: {e}")

In [ ]:
# ==========================================
# Quantitative Evaluation: Final Report
# ==========================================

# Calculate Averages
avg_v_hit = np.mean(quant_scores['vector']['hit']) * 100
avg_v_ndcg = np.mean(quant_scores['vector']['ndcg']) * 100
avg_h_hit = np.mean(quant_scores['hybrid']['hit']) * 100
avg_h_ndcg = np.mean(quant_scores['hybrid']['ndcg']) * 100

print("\n"  "="*40)
print("FINAL EVALUATION REPORT")
print("="*40)

# Print Table
print(f"{'Metric':<12} | {'Vector Only':<11} | {'GraphRAG (Hybrid)':<17} | {'Improvement'}")
print(f"{'-'*12}|{'-'*13}|{'-'*19}|{'-'*12}")
print(f"{'Hit Rate@10':<12} | {avg_v_hit:6.2f}%     | {avg_h_hit:6.2f}%            | {avg_h_hit - avg_v_hit:.2f}%p")
print(f"{'NDCG@10':<12} | {avg_v_ndcg:6.2f}%     | {avg_h_ndcg:6.2f}%            | {avg_h_ndcg - avg_v_ndcg:.2f}%p")
print("="*40)

# Conclusion
if avg_h_hit > avg_v_hit:
    print("Conclusion: GraphRAG outperforms Vector Search.")
else:
    print("Conclusion: Performance is similar. Check brand loyalty distribution.")

### Step 7. Evaluation: Qualitative RAGAS
* Quality and factual consistency of AI responses assessed via the Ragas framework (Judge: GPT-4o).
* **High Answer Relevancy (0.85)** achieved, confirming the model's ability to generate advice aligned with specific user needs.
* **Faithfulness (0.66)** observed, reflecting a balance between retrieved context and the model's persuasive generation capabilities.
* **Context Precision (0.24):** Measures if the ground truth is ranked highly in the retrieved context. (Formula: $S_{relevant} / Total_{retrieved}$)

In [ ]:
# ==========================================
# Qualitative Evaluation: RAGAS Settings
# ==========================================

SAMPLE_SIZE_RAGAS = 50
RAGAS_TOP_K = 3  # Contexts to retrieve

In [ ]:
# ==========================================
# Helper Functions
# ==========================================

def generate_ragas_answer(query, contexts):
    """
    Generate an answer using the Service Model (gpt-4o-mini).
    """
    context_block = "\n".join([f"- {c}" for c in contexts])
    prompt = f"""
    User Query: {query}
    
    Contexts:
    {context_block}
    
    Answer the user query based on the contexts provided. Recommend the best product.
    """
    # Use Global Client (gpt-4o-mini defined in Step 1)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
# ==========================================
# Execution: Data Generation & Judging (RAGAS)
# ==========================================

print(f"Connecting to Neo4j at {NEO4J_URI} (Using Global Driver)...")
driver.verify_connectivity()

# 1) Prepare test cases
test_cases = fetch_test_dataset(driver, sample_size=SAMPLE_SIZE_RAGAS)
print(f"Prepared {len(test_cases)} raw test cases.")

# 2) Generate synthetic queries (reduce leakage)
print("\nGenerating Synthetic Queries for RAGAS...")
for case in tqdm(test_cases, desc="Paraphrasing"):
    case["query_text"] = generate_synthetic_query(case["raw_review_text"])

data_samples = {
    "user_input": [],
    "response": [],
    "retrieved_contexts": [],
    "reference": [],  # IMPORTANT: string (RAGAS expects reference: str)
}

print("\nGenerating Answers and Contexts...")

for case in tqdm(test_cases, desc="Generating"):
    query_text = (case.get("query_text") or "").strip()
    if not query_text:
        continue

    # A) Embed query
    query_vec = get_embedding(query_text)

    # B) Retrieve contexts as strings (for RAGAS)
    contexts = run_hybrid_query(
        driver,
        query_embedding=query_vec,
        user_id=case.get("user_id"),
        loyalty_score=case.get("loyalty_score", 0.0),
        k=RAGAS_TOP_K,
        exclude_purchased=False,
        return_context_string=True
    )
    if not contexts:
        continue

    # C) Generate answer
    answer = generate_ragas_answer(query_text, contexts)

    # D) Append
    data_samples["user_input"].append(query_text)
    data_samples["response"].append(answer)
    data_samples["retrieved_contexts"].append(contexts)

    gt = case.get("target_product", "")
    data_samples["reference"].append(str(gt))

if len(data_samples["user_input"]) == 0:
    raise ValueError("No RAGAS samples generated (data_samples is empty).")

print("\nRunning RAGAS Evaluation (Judge: GPT-4o)...")

dataset = Dataset.from_dict(data_samples)

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
]

ragas_result = evaluate(
    dataset,
    metrics=metrics,
    llm=llm_judge,
    embeddings=emb
)

print("✔ RAGAS Evaluation Completed.")
print("[System] ragas_result exists:", "ragas_result" in globals())

In [ ]:
# ==========================================
# Qualitative Evaluation: Report (RAGAS)
# ==========================================

print("=" * 40)
print("RAGAS EVALUATION REPORT")
print("=" * 40)

df_results = ragas_result.to_pandas()

all_cols = df_results.columns.tolist()
print(f"[System] Detected Columns: {all_cols}")

target_metrics = ["faithfulness", "answer_relevancy", "context_precision"]
existing_metrics = [m for m in target_metrics if m in df_results.columns]

# Clean rows only if key metrics exist
cleanup_subset = [m for m in ["faithfulness", "answer_relevancy"] if m in existing_metrics]
df_clean = df_results.dropna(subset=cleanup_subset) if cleanup_subset else df_results

total_rows = len(df_results)
clean_count = len(df_clean)
missing_count = total_rows - clean_count

print(f"\n[Data Status]")
print(f"- Total Sample Count: {total_rows}")
print(f"- Analyzable Sample Count: {clean_count} (Success Rate: {clean_count/total_rows*100:.1f}%)")
print(f"- Missing/Failed Sample Count: {missing_count}")

print("-" * 40)
print("[Final Performance Evaluation Results (Mean)]")

if clean_count > 0 and existing_metrics:
    for metric in existing_metrics:
        print(f"- {metric}: {df_clean[metric].mean():.4f}")
else:
    print("No valid data available to calculate means.")


## Limitations & Future Work
- Benchmark against classical recommenders (e.g., MF / GNN-based CF) to quantify trade-offs.
- Enrich KG signals beyond brand loyalty (e.g., ingredients, price, category-level patterns).
